# Pandas vs PySpark

Notebook ini menerjemahkan operasi pandas yang sudah kamu kenal ke Spark.
Semua syntax yang dipakai di notebook ETL (`02_etl_superstore.ipynb`) dibahas di sini.

Data: Superstore, berupa 3 file CSV di folder `data/`:

| File | Isi |
| --- | --- |
| `superstore_orders.csv` | satu baris per item order: tanggal, ID customer, ID produk, lokasi, sales |
| `superstore_customers.csv` | ID customer, nama, segment |
| `superstore_products.csv` | ID produk, nama, kategori |

### Apa itu Spark?
- pandas bekerja di **1 komputer** dan menyimpan semua data di memori.
- Spark memecah data menjadi **partisi** dan memprosesnya **paralel**, sehingga bisa mengolah data yang terlalu besar untuk pandas.
- Kodenya sangat mirip pandas. Dua perbedaan terbesar:
  1. Spark itu **lazy**: baru menghitung saat kita meminta hasil (`show`, `count`, `write`).
  2. DataFrame Spark **tidak bisa diubah langsung**. Hasilnya selalu disimpan: `df = df.withColumn(...)`.

### SETUP

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

os.environ['PYSPARK_PYTHON'] = sys.executable   # Spark memakai Python dari env conda ini

spark = (SparkSession.builder
         .appName("pandas-vs-spark")
         .master("local[2]")                           # jalan di laptop, pakai 2 core
         .config("spark.driver.memory", "1g")          # RAM maksimal 1 GB
         .config("spark.sql.shuffle.partitions", "4")  # saat groupBy/join, data dibagi jadi 4 bagian
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")   # sembunyikan pesan warning Spark

Penjelasan setting di `SparkSession.builder`:

| Setting | Arti |
| --- | --- |
| `master("local[2]")` | Spark jalan di laptop sendiri dengan 2 core. Di cluster, isinya alamat cluster |
| `spark.driver.memory` | RAM maksimal yang boleh dipakai Spark |
| `spark.sql.shuffle.partitions` | Jumlah bagian data saat `groupBy`/`join`. Default 200, terlalu banyak untuk data kecil |
| `PYSPARK_PYTHON` | Memastikan Spark memakai Python dari env conda yang aktif |

- Saat pertama dijalankan, muncul beberapa baris `WARNING` ketika Spark start. Itu normal.
- `SparkSession` adalah pintu masuk Spark, seperti `import pandas as pd`.
- `functions as F` berisi fungsi-fungsi kolom Spark (`F.col`, `F.sum`, `F.trim`, ...).
- `Window` dipakai untuk membuat nomor urut (bagian 7).
- Selama session berjalan, Spark UI bisa dibuka di http://localhost:4040.

## 1. Membaca & melihat data

pandas: `pd.read_csv('file.csv')` → Spark: `spark.read.csv('file.csv', header=True, inferSchema=True, escape='"')`

- `header=True`: baris pertama adalah nama kolom (di pandas otomatis).
- `inferSchema=True`: Spark menebak tipe data tiap kolom (di pandas otomatis).
- `escape='"'`: beberapa teks berisi koma dan tanda kutip, misalnya `Chair, "Executive Red"`.
  pandas menanganinya otomatis; Spark butuh opsi ini, kalau tidak kolomnya bergeser.
- Path `../data/...` artinya folder `data` yang ada di sebelah folder `notebooks`.

In [ ]:
orders = spark.read.csv('../data/superstore_orders.csv', header=True, inferSchema=True, escape='"')

**Tampilkan beberapa baris** → pandas: `df.head()`

In [ ]:
orders.show(5)

### Kenapa harus `show()`?

Di pandas, menulis `df` di akhir cell langsung menampilkan tabel. Di Spark, menulis `orders` saja **tidak menampilkan data**,
hanya nama kolom dan tipenya:

In [ ]:
orders

Ini karena Spark **lazy**: DataFrame hanyalah "rencana", datanya belum dibaca atau dihitung.
Hal yang sama terjadi pada filter tanpa `show()`: hasilnya DataFrame baru, tapi belum ada data yang diproses.

In [ ]:
orders.filter(F.col('Sales') > 3000)

Data baru diproses saat kita memanggil **action**, misalnya:

| Action | Hasil |
| --- | --- |
| `show(n)` | menampilkan n baris sebagai teks |
| `count()` | jumlah baris |
| `toPandas()` | mengubah ke DataFrame pandas (hanya untuk data kecil) |
| `write...` | menyimpan ke file / database |

In [ ]:
orders.filter(F.col('Sales') > 3000).select('Order ID', 'Sales').show()

### Tampil otomatis seperti pandas

Spark bisa diatur supaya DataFrame langsung tampil sebagai tabel di Jupyter tanpa `show()`:

In [ ]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)     # tampil otomatis
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 5)     # maksimal 5 baris

orders

Praktis untuk eksplorasi, tapi perlu diingat:
- Setiap kali DataFrame tampil, Spark tetap menjalankan perhitungan di belakang layar.
- Ini hanya berlaku di Jupyter. Di script `.py` (misalnya yang dijalankan Airflow) tidak ada tampilan otomatis, jadi tetap pakai `show()`.

Di notebook ini kita kembali memakai `show()`, supaya sama dengan cara kerja di script:

In [ ]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", False)

**Jumlah baris dan kolom** → pandas: `df.shape`, `df.columns`

In [ ]:
print(orders.count(), len(orders.columns))
print(orders.columns)

**Tipe data tiap kolom** → pandas: `df.info()`

Selalu cek ini setelah membaca file: kolom angka harus `double`/`integer`, bukan `string`.
`Order Date` masih `string`, nanti diubah jadi tanggal (bagian 4).

In [ ]:
orders.printSchema()

## 2. Mengganti nama semua kolom

pandas: `df.columns = [...]` → Spark: `df.toDF(*nama_baru)`

Nama kolom dengan spasi (`Order ID`) merepotkan, terutama saat masuk ke database. Kita ubah ke **snake_case** (`order_id`).
Karena dipakai untuk 3 file, kita buat fungsi.

In [ ]:
def to_snake_case(df):
    new_names = [c.lower().replace(' ', '_').replace('-', '_').replace('/', '_') for c in df.columns]
    return df.toDF(*new_names)

orders = to_snake_case(orders)
orders.columns

Mulai sekarang semua kolom memakai nama snake_case.

## 3. Memilih, memfilter, mengurutkan

**Memilih kolom** → pandas: `df[['order_id', 'sales']]`

In [ ]:
orders.select('order_id', 'region', 'sales').show(3)

**Filter, 1 kondisi** → pandas: `df[df['sales'] > 3000]`

In [ ]:
orders.filter(F.col('sales') > 3000).select('order_id', 'state', 'sales').show()

**Filter, beberapa kondisi** → pandas: `df[(df['sales'] > 1000) & (df['region'] == 'West')]`. Setiap kondisi diberi kurung, sama seperti pandas.

In [ ]:
orders.filter((F.col('sales') > 1000) & (F.col('region') == 'West')).select('order_id', 'region', 'sales').show(5)

**Filter dengan teks** → pandas: `df.query('sales > 3000')`

In [ ]:
orders.filter('sales > 3000').count()

**Mengurutkan** → pandas: `df.sort_values('sales', ascending=False)`

In [ ]:
orders.orderBy(F.desc('sales')).select('order_id', 'product_id', 'sales').show(5)

**Frekuensi tiap nilai** → pandas: `df['region'].value_counts()`

In [ ]:
orders.groupBy('region').count().orderBy(F.desc('count')).show()

**Nilai unik** → pandas: `df['region'].unique()`

In [ ]:
orders.select('region').distinct().show()

`WEST` dan `West` adalah region yang sama tapi ditulis berbeda. Ini data kotor yang dibersihkan di bagian 4.

## 4. Membersihkan data

**Menambah / mengubah kolom** → pandas: `df['unit_price'] = df['sales'] / df['quantity']`

`withColumn(nama, nilai)` menambah kolom baru, atau mengganti kolom yang namanya sudah ada.
Hasilnya **harus disimpan** ke variabel, karena DataFrame Spark tidak bisa diubah langsung.

In [ ]:
with_price = orders.withColumn('unit_price', F.round(F.col('sales') / F.col('quantity'), 2))
with_price.select('sales', 'quantity', 'unit_price').show(3)

### Duplikat → pandas: `df.duplicated().sum()`, `df.drop_duplicates()`

In [ ]:
print('baris duplikat:', orders.count() - orders.dropDuplicates().count())

orders_unique = orders.dropDuplicates()
print(orders.count(), '->', orders_unique.count())

`dropDuplicates(['kolom'])` menyisakan **1 baris per nilai kolom itu** (pandas: `drop_duplicates('kolom')`).
Contoh: di file products, beberapa `product_id` punya 2 nama produk.

In [ ]:
products = to_snake_case(spark.read.csv('../data/superstore_products.csv', header=True, inferSchema=True, escape='"'))

products.groupBy('product_id').agg(F.countDistinct('product_name').alias('names')).filter('names > 1').show(3)
print(products.count(), '->', products.dropDuplicates(['product_id']).count())

### Nilai kosong → pandas: `df.isna().sum()`, `fillna`, `dropna`

In [ ]:
for c in orders.columns:
    n = orders.filter(F.col(c).isNull()).count()
    if n > 0:
        print(c, n)

In [ ]:
filled = orders.fillna({'postal_code': 0})     # isi nilai kosong (pandas: fillna)
dropped = orders.dropna(subset=['sales'])      # buang baris yang sales-nya kosong (pandas: dropna(subset=...))
print(orders.count(), '->', dropped.count())

### Teks → pandas: `.str.strip()`, `.str.title()`

In [ ]:
clean_region = orders.withColumn('region', F.initcap(F.trim('region')))
clean_region.select('region').distinct().show()

`F.trim` membuang spasi di awal/akhir, `F.initcap` membuat huruf pertama tiap kata kapital (`WEST` → `West`).

### Ubah tipe data → pandas: `.astype(str)`, `.str.len()`, `.str.zfill(5)`

Postal code terbaca sebagai angka, jadi `06824` berubah menjadi `6824` (angka 0 di depan hilang).
`cast('string')` mengubahnya jadi teks, `F.length` menghitung panjang teks, `F.lpad` menambah `0` di kiri sampai 5 karakter.

In [ ]:
zip_fixed = orders.withColumn('postal_code', F.lpad(F.col('postal_code').cast('string'), 5, '0'))

orders.filter(F.length(F.col('postal_code').cast('string')) < 5).select('city', 'postal_code').show(3)
zip_fixed.filter(F.col('city') == 'Fairfield').select('city', 'postal_code').show(3)

### Tanggal → pandas: `pd.to_datetime`, `.dt.year`, `.dt.strftime`

- `F.to_date(kolom, 'M/d/yyyy')` mengubah teks `11/8/2020` menjadi tanggal. Polanya: `M` = bulan, `d` = hari, `yyyy` = tahun.
- `F.year`, `F.quarter`, `F.month` mengambil bagian tanggal.
- `F.date_format(kolom, pola)` mengubah tanggal menjadi teks dengan format tertentu.

In [ ]:
dated = orders.withColumn('order_date', F.to_date('order_date', 'M/d/yyyy'))

(dated.select('order_date',
              F.year('order_date').alias('year'),
              F.quarter('order_date').alias('quarter'),
              F.month('order_date').alias('month'),
              F.date_format('order_date', 'MMMM').alias('month_name'),
              F.date_format('order_date', 'EEEE').alias('day_name'),
              F.date_format('order_date', 'yyyyMMdd').cast('int').alias('date_id'))
      .show(3))

## 5. Group by & agregasi

pandas: `df.groupby('region').agg(total_sales=('sales', 'sum'), rows=('order_id', 'count'))`
→ Spark: `df.groupBy('region').agg(F.sum('sales').alias('total_sales'), ...)`

`.alias()` memberi nama kolom hasil.

In [ ]:
(clean_region.groupBy('region')
    .agg(F.round(F.sum('sales'), 2).alias('total_sales'),
         F.count('order_id').alias('rows'),
         F.countDistinct('customer_id').alias('customers'))
    .orderBy(F.desc('total_sales'))
    .show())

## 6. Menggabungkan data dari beberapa file (join)

`orders` hanya berisi `product_id` dan `customer_id`. Nama produk, kategori, dan nama customer ada di file lain.

pandas: `pd.merge(orders, products, on='product_id', how='left')` → Spark: `orders.join(products, on='product_id', how='left')`

In [ ]:
customers = to_snake_case(spark.read.csv('../data/superstore_customers.csv', header=True, inferSchema=True, escape='"'))

orders_products = orders.join(products, on='product_id', how='left')
orders_products.select('order_id', 'product_id', 'product_name', 'category', 'sales').show(3, truncate=False)

**Join 3 tabel**: sambungkan `.join()` berurutan, seperti menyambung `pd.merge()`.

Perhatikan jumlah barisnya: hasil join **lebih banyak** dari `orders`. File customers dan products berisi ID duplikat,
dan join mengulang order untuk setiap baris yang cocok. Selalu cek jumlah baris setelah join, dan bersihkan duplikat sebelum join.

In [ ]:
full = (orders
        .join(customers, on='customer_id', how='left')
        .join(products, on='product_id', how='left'))
print('orders:', orders.count(), '| setelah join:', full.count())

full_clean = (orders
              .join(customers.dropDuplicates(['customer_id']), on='customer_id', how='left')
              .join(products.dropDuplicates(['product_id']), on='product_id', how='left'))
print('setelah duplikat dibersihkan:', full_clean.count())

**Jenis join** (sama dengan `how=` di pandas dan JOIN di SQL):

| `how=` | Yang disimpan |
| --- | --- |
| `'inner'` | hanya baris yang cocok di kedua tabel |
| `'left'` | semua baris tabel kiri |
| `'left_anti'` | baris tabel kiri yang **tidak punya** pasangan (untuk mencari data yang hilang) |

`left_anti`: order yang produknya tidak ada di file products.

In [ ]:
orders.join(products, on='product_id', how='left_anti').select('order_id', 'product_id').show()

`inner`: hanya order yang produknya ada. Dua order di atas ikut terbuang.

In [ ]:
valid = orders.join(products.dropDuplicates(['product_id']).select('product_id'), on='product_id', how='inner')
print(orders.count(), '->', valid.count())

Join juga bisa memakai **beberapa kolom** sekaligus: `on=['state', 'city']` (pandas: `on=['state', 'city']`).
Baris dianggap cocok kalau semua kolomnya sama.

## 7. Menumpuk baris & membuat nomor urut

**Menumpuk 2 DataFrame** → pandas: `pd.concat([a, b])` → Spark: `a.union(b)`. Kolomnya harus sama.

In [ ]:
order_dates = dated.select(F.col('order_date').alias('full_date'))
ship_dates = orders.select(F.to_date('ship_date', 'M/d/yyyy').alias('full_date'))

all_dates = order_dates.union(ship_dates).distinct()
print(order_dates.count(), '+', ship_dates.count(), '-> unik:', all_dates.count())

**Nomor urut 1, 2, 3, ...** → pandas: `df = df.sort_values(...)`, lalu `df['id'] = range(1, len(df) + 1)`

Spark: `F.row_number().over(Window.orderBy(...))` memberi nomor urut sesuai urutan kolom yang dipilih.
Dipakai untuk membuat ID bagi data yang belum punya ID.

In [ ]:
cities = orders.select('state', 'city').dropDuplicates()
cities_with_id = cities.withColumn('city_id', F.row_number().over(Window.orderBy('state', 'city')))
cities_with_id.show(5)

## Ringkasan pandas vs Spark

| pandas | Spark |
| --- | --- |
| `pd.read_csv(path)` | `spark.read.csv(path, header=True, inferSchema=True, escape='"')` |
| `df.head()` | `df.show(5)` |
| `df.shape` | `(df.count(), len(df.columns))` |
| `df.info()` | `df.printSchema()` |
| `df.columns = [...]` | `df = df.toDF(*nama_baru)` |
| `df[['a', 'b']]` | `df.select('a', 'b')` |
| `df[df['a'] > 1]` / `df.query('a > 1')` | `df.filter(F.col('a') > 1)` / `df.filter('a > 1')` |
| `df.sort_values('a', ascending=False)` | `df.orderBy(F.desc('a'))` |
| `df['a'].value_counts()` | `df.groupBy('a').count()` |
| `df['a'].unique()` | `df.select('a').distinct()` |
| `df['new'] = ...` | `df = df.withColumn('new', ...)` |
| `df.drop_duplicates()` / `(subset='a')` | `df.dropDuplicates()` / `(['a'])` |
| `df.isna().sum()` | `df.filter(F.col('a').isNull()).count()` |
| `df.fillna({...})`, `df.dropna(subset=[...])` | `df.fillna({...})`, `df.dropna(subset=[...])` |
| `.str.strip().str.title()` | `F.initcap(F.trim('a'))` |
| `.astype(str)`, `.str.len()`, `.str.zfill(5)` | `.cast('string')`, `F.length()`, `F.lpad(..., 5, '0')` |
| `pd.to_datetime(a, format='%m/%d/%Y')` | `F.to_date('a', 'M/d/yyyy')` |
| `.dt.year`, `.dt.quarter`, `.dt.month` | `F.year()`, `F.quarter()`, `F.month()` |
| `.dt.strftime('%Y%m%d')` | `F.date_format('a', 'yyyyMMdd')` |
| `df.groupby('a').agg(...)` | `df.groupBy('a').agg(F.sum('b').alias('total'))` |
| `pd.merge(a, b, on='id', how='left')` | `a.join(b, on='id', how='left')` |
| `pd.concat([a, b])` | `a.union(b)` |
| `df['id'] = range(1, len(df) + 1)` | `F.row_number().over(Window.orderBy('a'))` |
| `df.to_sql(...)`, `pd.read_sql(...)` | `df.write.jdbc(...)`, `spark.read.jdbc(...)` (dipakai di notebook ETL) |

Selesai. Hentikan Spark supaya memori laptop dilepas.

In [ ]:
spark.stop()